# Load and prepare the dataset

## Import Module

In [93]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score, r2_score
)
from sklearn.inspection import permutation_importance

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## Load Dataset

In [94]:
df = pd.read_parquet('jkp_data_us.parquet')

print(df.head(10))

        id        eom excntry  ret_exc_lead1m         me     be_me     at_me  \
0  10001.0 2008-07-31     USA       -0.021006  44.098861  0.712921  1.299920   
1  10001.0 2008-10-31     USA       -0.130349  36.096701  0.856006  1.617239   
2  10001.0 2008-11-30     USA        0.155960  31.225819  0.989534  1.869511   
3  10001.0 2008-12-31     USA        0.034117  35.493221  0.870561  1.644737   
4  10001.0 2009-01-31     USA        0.056075  36.533093       NaN       NaN   
5  10001.0 2009-02-28     USA       -0.080696  38.415178       NaN       NaN   
6  10001.0 2009-03-31     USA        0.044496  35.174001       NaN       NaN   
7  10001.0 2009-04-30     USA        0.002924  36.550000  0.829603  2.074391   
8  10001.0 2009-05-31     USA        0.019371  36.921918  0.821247  2.053496   
9  10001.0 2009-06-30     USA       -0.047201  36.988171  0.819776  2.049817   

    sale_me     ni_me    ocf_me  ...   roe_ch5   roa_ch5  cfoa_ch5  gmar_ch5  \
0  1.346361  0.051180 -0.028822  ...  0

In [95]:
print(df.sort_values(by='eom',ascending=False).head(5))

             id        eom excntry  ret_exc_lead1m             me     be_me  \
78970   13706.0 2024-12-31     USA             NaN     745.254671  0.780830   
129753  15056.0 2024-12-31     USA             NaN    4053.899760  0.046441   
609444  84373.0 2024-12-31     USA             NaN   16838.238968  0.392975   
852491  91233.0 2024-12-31     USA             NaN  479583.112431  0.016177   
426068  72996.0 2024-12-31     USA             NaN   10857.606427  0.434093   

           at_me   sale_me     ni_me    ocf_me  ...   roe_ch5   roa_ch5  \
78970   0.866234  0.291506 -0.068328 -0.132321  ...  0.188890  0.102250   
129753  0.092849  0.081463  0.008582  0.018010  ...  1.452276  0.507386   
609444  0.824374  0.555046  0.049886  0.074117  ... -0.003115  0.000140   
852491  0.088270  0.055027  0.025560  0.025382  ...  0.265526  0.018323   
426068  3.482285  0.510115  0.051532  0.009501  ... -0.017933 -0.002710   

        cfoa_ch5    gmar_ch5   cash_me  netis_mev  divspc1m_me  divspc12m_

## Check Shape of Dataset

In [96]:
dfColumns = list(df.columns)
print(dfColumns)

['id', 'eom', 'excntry', 'ret_exc_lead1m', 'me', 'be_me', 'at_me', 'sale_me', 'ni_me', 'ocf_me', 'fcf_me', 'ebitda_mev', 'bev_mev', 'eq_dur', 'ival_me', 'netdebt_me', 'rd_me', 'debt_me', 'div12m_me', 'eqpo_me', 'eqnpo_me', 'gp_at', 'ope_be', 'ni_be', 'cop_at', 'op_at', 'ocf_at', 'ebit_sale', 'gp_atl1', 'ope_bel1', 'cop_atl1', 'niq_be', 'niq_at', 'pi_nix', 'op_atl1', 'ocf_at_chg1', 'niq_be_chg1', 'ret_1_0', 'ret_2_0', 'ret_3_0', 'ret_3_1', 'ret_6_0', 'ret_6_1', 'ret_9_0', 'ret_9_1', 'ret_12_0', 'ret_12_1', 'ret_12_7', 'ret_60_12', 'ret_18_1', 'ret_24_1', 'ret_24_12', 'ret_36_1', 'ret_36_12', 'ret_48_1', 'ret_60_1', 'ret_60_36', 'resff3_12_1', 'resff3_6_1', 'seas_1_1an', 'seas_1_1na', 'seas_2_5an', 'seas_2_5na', 'seas_6_10an', 'seas_6_10na', 'seas_11_15an', 'seas_11_15na', 'seas_16_20an', 'seas_16_20na', 'at_gr1', 'sale_gr1', 'capx_gr1', 'inv_gr1', 'noa_gr1a', 'ppeinv_gr1a', 'lnoa_gr1a', 'debt_gr3', 'sale_gr3', 'capx_gr3', 'emp_gr1', 'sale_emp_gr1', 'at_be', 'capx_gr2', 'inv_gr1a', 'lti_

In [97]:
print("Shape of Dataset:", df.shape)
print("")
print("Observations No.:", df.shape[0])
print("")
print("Features No.:", df.shape[1])

Shape of Dataset: (961836, 201)

Observations No.: 961836

Features No.: 201


In [98]:
id = df["id"].unique()
print("No. of Stock:", len(id))

No. of Stock: 13213


## Define Features Set & Target

In [99]:
# Candidate prediction variables for first-stage screening and pooled models

prediction_features = [
    'ret_2_0', 'ret_12_0', 'ret_24_12', 'resff3_12_1', #Momentum
    'aliq_at', 'ami_126d', 'turnover_126d', 'bidaskhl_21d', #Liquidity/Frictions
    'ivol_capm_21d', 'ivol_ff3_21d', 'rvol_21d', 'beta_252d', #Risk/Volatilitye
    'gp_at', 'ocf_at', 'qmj', 'f_score', #Profitability/Quality
    'be_me', 'at_gr1', 'sale_gr1', 'capex_abn', 'me', #Value/Investment
    'netdebt_me', 'o_score', 'z_score', 'eqnetis_at' #Distress/Issuance
]

target = 'ret_exc_lead1m'

In [100]:
# Cluster variables are used only for k-means firm-type assignment

cluster_features = [
    'age', #Lifecycle
    'gp_at', 'ocf_at', 'sale_gr1', 'inv_gr1', 'dsale_dinv', #Profitability–Investment structure
    'netdebt_me', 'kz_index', 'z_score', #Financial fragility
    'niq_su', 'ni_inc8q' #Information environment
]

In [101]:
# Union of prediction and clustering variables; duplicates removed while preserving order

features = list(dict.fromkeys(prediction_features + cluster_features))

In [102]:
# Positive, highly skewed variables chosen for log1p transform

log_cols = ["me", "age"]

for c in log_cols:
    print("No. of Zero of ",c,":",(df[c]==0).sum())

No. of Zero of  me : 0
No. of Zero of  age : 12


## Drop Empty Target

In [103]:
df[target].isnull().mean()*100
df_clear = df.dropna(subset=['id', 'eom', target])
df_clear[['id', 'eom', target]].isnull().mean()*100

id                0.0
eom               0.0
ret_exc_lead1m    0.0
dtype: float64

In [104]:
(df_clear[features].isnull().mean()*100).sort_values()

age               0.000000
me                0.112257
bidaskhl_21d      0.521187
ret_2_0           0.695868
ivol_capm_21d     1.236734
ivol_ff3_21d      1.236734
rvol_21d          1.236734
turnover_126d     1.920116
ami_126d          2.364067
netdebt_me        2.726231
ocf_at            3.197055
gp_at             3.232605
eqnetis_at        3.711999
beta_252d         4.083263
at_gr1            4.102942
ret_12_0          6.405114
be_me             6.788122
sale_gr1          8.460235
niq_su           11.913969
ret_24_12        13.026596
kz_index         13.517946
resff3_12_1      13.875137
aliq_at          14.640624
ni_inc8q         15.731772
z_score          16.689185
o_score          16.759650
capex_abn        20.998655
qmj              21.411288
f_score          23.108371
inv_gr1          32.568764
dsale_dinv       34.276110
dtype: float64

In [105]:
print(f"Data Time Range: {df_clear['eom'].min().date()} to {df_clear['eom'].max().date()}")

Data Time Range: 2005-01-31 to 2024-11-30


## Change Data Type and Drop Unselected Features

In [106]:
df_clear['eom'] = pd.to_datetime(df_clear['eom'])
df_clear = df_clear.sort_values(['id', 'eom']).reset_index(drop=True)

In [107]:
print(df_clear['eom'][0])

2008-07-31 00:00:00


In [108]:
df_clear = df_clear[['id', 'eom', target] + features].copy()
print(df_clear.head(5))

# missing_features = []
# for c in prediction_features+cluster_features:
#     if c not in df_clear.columns:
#         missing_features.append(c)
# print(missing_features)

        id        eom  ret_exc_lead1m   ret_2_0  ret_12_0  ret_24_12  \
0  10001.0 2008-07-31       -0.021006 -0.062580  0.133622   0.408844   
1  10001.0 2008-10-31       -0.130349 -0.153611 -0.014414   0.256010   
2  10001.0 2008-11-30        0.155960 -0.146495 -0.205725   0.289406   
3  10001.0 2008-12-31        0.034117  0.005571 -0.078204   0.334092   
4  10001.0 2009-01-31        0.056075  0.195455 -0.040900   0.295813   

   resff3_12_1   aliq_at     ami_126d turnover_126d  ... netdebt_me   o_score  \
0    -0.013293  0.582519  26.74428303    0.00185593  ...   0.236605 -2.484459   
1    -0.065312  0.667072   1.80525519    0.00196731  ...   0.327620 -2.484459   
2    -0.158375  0.667072   2.00651494    0.00117302  ...   0.378725 -2.484459   
3    -0.230158  0.667072   1.95548890    0.00124243  ...   0.333190 -2.484459   
4    -0.227538       NaN   2.01399288    0.00117742  ...        NaN       NaN   

    z_score eqnetis_at    age   inv_gr1  dsale_dinv  kz_index  niq_su  \
0  3.22

In [109]:
print(df_clear.dtypes.sort_values())
object_f = []
for f in features:
    if df_clear[f].dtypes == 'object':
        object_f.append(f)
print(object_f)
print(df_clear[object_f].head(5))
for f in features:
    df_clear[f] = pd.to_numeric(df_clear[f], errors='coerce')
print(df_clear.dtypes.sort_values())
print(df_clear[object_f].head(5))


id                       float64
kz_index                 float64
dsale_dinv               float64
inv_gr1                  float64
age                      float64
eqnetis_at               float64
z_score                  float64
o_score                  float64
netdebt_me               float64
me                       float64
capex_abn                float64
sale_gr1                 float64
at_gr1                   float64
be_me                    float64
f_score                  float64
qmj                      float64
ocf_at                   float64
gp_at                    float64
eom               datetime64[ms]
ret_exc_lead1m           float64
ret_2_0                  float64
ret_12_0                 float64
ret_24_12                float64
resff3_12_1              float64
niq_su                   float64
aliq_at                  float64
ni_inc8q                 float64
turnover_126d             object
bidaskhl_21d              object
ivol_capm_21d             object
ivol_ff3_2

## Data Split

In [110]:
# def split_train_valid_test(df: pd.DataFrame, validation_cutoff, test_cutoff):
#     """
#     Official project split:
#     Train: 2005-01 to 2015-12
#     Valid: 2016-01 to 2018-12
#     Test:  2019-01 to 2024-12
#     """
#     train_mask = (df['eom'] < validation_cutoff)
#     valid_mask = (df['eom'] >= validation_cutoff) & (df['eom'] < test_cutoff)
#     test_mask  = (df['eom'] >= test_cutoff)

#     train_df = df.loc[train_mask].copy()
#     valid_df = df.loc[valid_mask].copy()
#     test_df  = df.loc[test_mask].copy()

#     return train_df, valid_df, test_df

def split_df_2(df: pd.DataFrame, cutoff, end):

    # Dictionary: refit_date -> (estimation_window, next_6_month_prediction_block)

    train_mask = (df['eom'] < cutoff)
    valid_mask = (df['eom'] >= cutoff) & (df['eom'] < end)

    train_df = df.loc[train_mask].copy()
    valid_df = df.loc[valid_mask].copy()

    return train_df, valid_df

In [111]:
# Inner split used only for model development:
# inner train fits candidate models; inner validation tunes K, alpha, and lambda.
# This keeps the outer validation period cleaner for finalized model comparison.

inner_train_df, inner_valid_df = split_df_2(df_clear, '2013-01-01', '2016-01-01')

In [112]:
def expanding_window_6_months(date):
    if date[6] == '1':
        new_date = str(int(date[0:4]))+'-'+'07'+date[7:]
    if date[6] == '7':
        new_date = str(int(date[0:4])+1)+'-'+'01'+date[7:]
    return new_date

def date_updater(date=pd.to_datetime('2019-01-01'), frequency=1, unit='months'):

    """
    unit: days, weeks, months, years
    
    e.g. 1 months, 6 months
    """

    date = pd.to_datetime(date)

    if unit == 'days':
        date += pd.Timedelta(days=frequency)

    if unit == 'weeks':
        date += pd.Timedelta(weeks=frequency)
    
    if unit == 'months':
        date += pd.DateOffset(months=frequency)
    
    if unit == 'years':
        date += pd.DateOffset(years=frequency)

    return date

# Build 6-month expanding-window refit blocks:
# each entry contains (estimation window up to refit date (start_date), next 6-month prediction block).

def expanding_window(df, start_date, end_date, frequency=6, unit='months'):

    update_set = {} # Today: (train_df, pred_df)

    while start_date < end_date:
        next_date = date_updater(start_date, frequency, unit)
        update_set[start_date] = split_df_2(df, start_date, next_date)
        start_date = next_date
    
    return update_set

In [113]:
# Build 6-month expanding-window refit blocks:
# each entry contains (estimation window up to refit date, next 6-month prediction block).

valid_update_set = expanding_window(df_clear, pd.to_datetime('2016-01-01'), pd.to_datetime('2019-01-01'), 6, 'months')
test_update_set = expanding_window(df_clear, pd.to_datetime('2019-01-01'), pd.to_datetime('2025-01-01'), 6, 'months')

In [114]:
def summarize_split(name, df):
    print(f"{name}: {df['eom'].min().date()} to {df['eom'].max().date()}, rows={len(df):,}, ids={df['id'].nunique():,}")

summarize_split("Inner train", inner_train_df)
summarize_split("Inner valid", inner_valid_df)
print("")

for k, (train, valid) in valid_update_set.items():
    summarize_split(f"Train {k}", train)
    summarize_split(f"Valid {k}", valid)
    print("")

for k, (train, test) in test_update_set.items():
    summarize_split(f"Train {k}", train)
    summarize_split(f"Test  {k}", test)
    print("")

Inner train: 2005-01-31 to 2012-12-31, rows=383,570, ids=6,869
Inner valid: 2013-01-31 to 2015-12-31, rows=133,564, ids=4,846

Train 2016-01-01 00:00:00: 2005-01-31 to 2015-12-31, rows=517,134, ids=7,842
Valid 2016-01-01 00:00:00: 2016-01-31 to 2016-06-30, rows=22,602, ids=3,962

Train 2016-07-01 00:00:00: 2005-01-31 to 2016-06-30, rows=539,736, ids=7,912
Valid 2016-07-01 00:00:00: 2016-07-31 to 2016-12-31, rows=22,055, ids=3,903

Train 2017-01-01 00:00:00: 2005-01-31 to 2016-12-31, rows=561,791, ids=8,038
Valid 2017-01-01 00:00:00: 2017-01-31 to 2017-06-30, rows=21,882, ids=3,846

Train 2017-07-01 00:00:00: 2005-01-31 to 2017-06-30, rows=583,673, ids=8,167
Valid 2017-07-01 00:00:00: 2017-07-31 to 2017-12-31, rows=22,017, ids=3,889

Train 2018-01-01 00:00:00: 2005-01-31 to 2017-12-31, rows=605,690, ids=8,310
Valid 2018-01-01 00:00:00: 2018-01-31 to 2018-06-30, rows=21,836, ids=3,855

Train 2018-07-01 00:00:00: 2005-01-31 to 2018-06-30, rows=627,526, ids=8,469
Valid 2018-07-01 00:00:00:

## Data Preprocessing

In [115]:
# Custom preprocessing pipeline:
# fit all preprocessing objects on the training segment only, then apply them to later blocks.
# This avoids look-ahead bias in winsorization, imputation, and standardization.

class StockPreprocessor:
    """
    Preprocessing pipeline for stock characteristics.

    Steps:
    1. impute missing values with training medians
    2. log(1+x) transform selected variables
    3. winsorize using training-set quantiles only
    4. standardize using training mean/std
    """

    def __init__(
        self,
        feature_cols,
        target_col,
        log1p_cols=None,
        id_col = 'id',
        date_col = 'eom',
        winsor_lower=0.01,
        winsor_upper=0.99
    ):
        self.id_col = id_col
        self.date_col = date_col
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.log1p_cols = log1p_cols if log1p_cols is not None else []
        self.winsor_lower = winsor_lower
        self.winsor_upper = winsor_upper

        # These objects are learned from training data only
        self.X_lower_bounds_ = None
        self.X_upper_bounds_ = None
        self.y_lower_bounds_ = None
        self.y_upper_bounds_ = None
        self.imputer_ = None
        self.scaler_ = None

    def _apply_log1p(self, df):
        
        # Apply log1p only to selected positive, highly skewed variables.
        # Their transformed names become log_<variable> in the processed output.

        out = df.copy()

        for col in self.log1p_cols:
            if col in out.columns:
                out[col] = np.log1p(out[col])
        
        return out

    def _winsorize_with_fitted_bounds(self, df):
        
        # Winsorization bounds are estimated from the training segment only
        # and then applied transformation to the corresponding validation / test block.

        X_wins = np.clip(df[self.feature_cols].values.astype(float), self.X_lower_bounds_, self.X_upper_bounds_)

        y_wins = df[self.target_col].clip(self.y_lower_bounds_, self.y_upper_bounds_)

        return X_wins, y_wins

    def fit(self, train_df):
        """
        Fit preprocessing objects using training data only.
        """
        X = train_df[self.feature_cols].copy()
        y = train_df[self.target_col].copy()

        # Step 1: fit median imputer on training only
        self.imputer_ = SimpleImputer(strategy="median")

        X_imputed = pd.DataFrame(self.imputer_.fit_transform(X), columns=self.feature_cols, index=train_df.index)
        X_imputed = self._apply_log1p(X_imputed)

        # Step 2: compute winsorization bounds from training only
        self.X_lower_bounds_ = X_imputed.quantile(self.winsor_lower).values.astype(float)
        self.X_upper_bounds_ = X_imputed.quantile(self.winsor_upper).values.astype(float)

        self.y_lower_bounds_ = float(y.quantile(self.winsor_lower))
        self.y_upper_bounds_ = float(y.quantile(self.winsor_upper))

        df = pd.concat([y, X_imputed], axis=1)

        # Apply winsorization to the training data
        X, y = self._winsorize_with_fitted_bounds(df)

        # Step 3: fit scaler on training only
        self.scaler_ = StandardScaler()
        self.scaler_.fit(X)

        return self

    def transform(self, df):
        """
        Apply fitted preprocessing objects to any new dataset
        such as validation or test.
        """
        X = df[self.feature_cols].copy()
        y = df[self.target_col].copy()

        # Apply the same training-fitted median imputer
        X_imputed = pd.DataFrame(self.imputer_.transform(X), columns=self.feature_cols, index=df.index)
        X_imputed = self._apply_log1p(X_imputed)
        out = pd.concat([y, X_imputed], axis=1)

        # Apply the same winsorization bounds learned from training
        X, y = self._winsorize_with_fitted_bounds(out)

        # Apply the same training-fitted scaler
        X_scaled = self.scaler_.transform(X)

        # Put transformed features back into a DataFrame
        X_out = pd.DataFrame(X_scaled, columns=self.feature_cols, index=out.index)

        # Return original dataset with transformed feature columns replaced
        out = pd.concat([df[[self.id_col, self.date_col]].copy(), y, X_out], axis=1)

        # After preprocessing, 'me' and 'age' become 'log_me' and 'log_age'
        for x in self.log1p_cols:
            if x in out.columns:
                out = out.rename(columns={x: f"log_{x}"})
        return out

    def fit_transform(self, train_df):
        """
        Convenience method: fit on training data and transform training data.
        """
        self.fit(train_df)
        return self.transform(train_df)

## Preprocessing Data

In [116]:
# Fit preprocessing objects on inner training data only.

SP_inner = StockPreprocessor(feature_cols=features, target_col=target, log1p_cols=log_cols)
SP_inner.fit(inner_train_df)

In [117]:
# Apply the same fitted transforms to inner validation to avoid look-ahead bias.

inner_train_df = SP_inner.transform(inner_train_df)
inner_valid_df = SP_inner.transform(inner_valid_df)


In [118]:
# All downstream model code should use model_features, not raw features

model_features = features.copy()
for i in log_cols:
    model_features[model_features.index(i)] = f"log_{i}"
print(model_features)

['ret_2_0', 'ret_12_0', 'ret_24_12', 'resff3_12_1', 'aliq_at', 'ami_126d', 'turnover_126d', 'bidaskhl_21d', 'ivol_capm_21d', 'ivol_ff3_21d', 'rvol_21d', 'beta_252d', 'gp_at', 'ocf_at', 'qmj', 'f_score', 'be_me', 'at_gr1', 'sale_gr1', 'capex_abn', 'log_me', 'netdebt_me', 'o_score', 'z_score', 'eqnetis_at', 'log_age', 'inv_gr1', 'dsale_dinv', 'kz_index', 'niq_su', 'ni_inc8q']


In [119]:
log_prediction_features = prediction_features.copy()
for i in log_cols:
    if i in log_prediction_features:
        log_prediction_features[log_prediction_features.index(i)] = f"log_{i}"

log_cluster_features = cluster_features.copy()
for i in log_cols:
    if i in log_cluster_features:
        log_cluster_features[log_cluster_features.index(i)] = f"log_{i}"

print(log_prediction_features)
print(log_cluster_features)

['ret_2_0', 'ret_12_0', 'ret_24_12', 'resff3_12_1', 'aliq_at', 'ami_126d', 'turnover_126d', 'bidaskhl_21d', 'ivol_capm_21d', 'ivol_ff3_21d', 'rvol_21d', 'beta_252d', 'gp_at', 'ocf_at', 'qmj', 'f_score', 'be_me', 'at_gr1', 'sale_gr1', 'capex_abn', 'log_me', 'netdebt_me', 'o_score', 'z_score', 'eqnetis_at']
['log_age', 'gp_at', 'ocf_at', 'sale_gr1', 'inv_gr1', 'dsale_dinv', 'netdebt_me', 'kz_index', 'z_score', 'niq_su', 'ni_inc8q']


In [120]:
# At each refit date, we re-estimate preprocessing objects, centroids, and coefficients
# using only data available up to that date, then forecast the next 6 months.

# Dictionary of 6-month expanding-window refit blocks for the validation stage:
# key = refit date; value = (past estimation sample, next 6-month forecast block)

valid_update_window = {}
for start_date in valid_update_set.keys():
    SP = StockPreprocessor(feature_cols=features, target_col=target, log1p_cols=log_cols)
    SP.fit(valid_update_set[start_date][0])
    valid_update_window[start_date] = [SP.transform(valid_update_set[start_date][0]),SP.transform(valid_update_set[start_date][1])]

In [121]:
# Dictionary of 6-month expanding-window refit blocks for the test stage:
# key = refit date; value = (past estimation sample, next 6-month forecast block)

test_update_window = {}
for start_date in test_update_set.keys():
    SP = StockPreprocessor(feature_cols=features, target_col=target, log1p_cols=log_cols)
    SP.fit(test_update_set[start_date][0])
    test_update_window[start_date] = [SP.transform(test_update_set[start_date][0]),SP.transform(test_update_set[start_date][1])]

## Check Preprocessed Data

In [122]:
mean_pre_valid = 0
for x in valid_update_window.keys():
    mean_pre_valid += valid_update_window[x][0][model_features].mean()

print("Train-Valid Set Mean:\n", mean_pre_valid/len(valid_update_window), sep='')

Train-Valid Set Mean:
ret_2_0         -7.048357e-18
ret_12_0         7.868081e-18
ret_24_12       -5.276158e-17
resff3_12_1     -4.725758e-18
aliq_at          6.491891e-17
ami_126d         3.294575e-18
turnover_126d    1.525350e-16
bidaskhl_21d     1.271500e-16
ivol_capm_21d   -1.728792e-17
ivol_ff3_21d    -3.803385e-17
rvol_21d        -6.971355e-17
beta_252d        2.232588e-17
gp_at            3.236308e-17
ocf_at           1.127508e-17
qmj              3.154101e-17
f_score         -6.738544e-18
be_me           -2.433274e-16
at_gr1           1.262577e-17
sale_gr1         2.014461e-17
capex_abn       -2.739318e-18
log_me           7.733919e-17
netdebt_me      -1.083197e-17
o_score          1.109048e-16
z_score         -1.369487e-16
eqnetis_at       1.105065e-17
log_age          1.968154e-17
inv_gr1         -2.952456e-17
dsale_dinv       2.689864e-18
kz_index         1.276347e-17
niq_su          -1.333372e-18
ni_inc8q        -3.897035e-18
dtype: float64


In [123]:
std_pre_valid = 0
for x in valid_update_window.keys():
    std_pre_valid += valid_update_window[x][0][model_features].std(ddof=0)

print("Train-Valid Set Std:\n", std_pre_valid/len(valid_update_window), sep='')

Train-Valid Set Std:
ret_2_0          1.0
ret_12_0         1.0
ret_24_12        1.0
resff3_12_1      1.0
aliq_at          1.0
ami_126d         1.0
turnover_126d    1.0
bidaskhl_21d     1.0
ivol_capm_21d    1.0
ivol_ff3_21d     1.0
rvol_21d         1.0
beta_252d        1.0
gp_at            1.0
ocf_at           1.0
qmj              1.0
f_score          1.0
be_me            1.0
at_gr1           1.0
sale_gr1         1.0
capex_abn        1.0
log_me           1.0
netdebt_me       1.0
o_score          1.0
z_score          1.0
eqnetis_at       1.0
log_age          1.0
inv_gr1          1.0
dsale_dinv       1.0
kz_index         1.0
niq_su           1.0
ni_inc8q         1.0
dtype: float64


In [124]:
for x in valid_update_window.keys():
    print('Shape of train-valid set', x,':', valid_update_window[x][0].shape,'/', valid_update_window[x][1].shape)

Shape of train-valid set 2016-01-01 00:00:00 : (517134, 34) / (22602, 34)
Shape of train-valid set 2016-07-01 00:00:00 : (539736, 34) / (22055, 34)
Shape of train-valid set 2017-01-01 00:00:00 : (561791, 34) / (21882, 34)
Shape of train-valid set 2017-07-01 00:00:00 : (583673, 34) / (22017, 34)
Shape of train-valid set 2018-01-01 00:00:00 : (605690, 34) / (21836, 34)
Shape of train-valid set 2018-07-01 00:00:00 : (627526, 34) / (22303, 34)


In [125]:
mean_pre_test = 0
for x in test_update_window.keys():
    mean_pre_test += test_update_window[x][0][model_features].mean()

print("Train-Test Set Mean:\n", mean_pre_test/len(test_update_window), sep='')

Train-Test Set Mean:
ret_2_0         -1.217620e-18
ret_12_0         4.735510e-18
ret_24_12        2.635694e-17
resff3_12_1     -3.809112e-18
aliq_at          5.245312e-17
ami_126d        -2.318052e-17
turnover_126d   -1.240916e-17
bidaskhl_21d    -1.612159e-16
ivol_capm_21d    7.453257e-17
ivol_ff3_21d     1.207837e-17
rvol_21d         5.930699e-17
beta_252d       -1.379004e-17
gp_at            2.153519e-17
ocf_at          -8.241747e-18
qmj             -5.371744e-18
f_score          4.390001e-17
be_me            2.958650e-17
at_gr1          -7.400733e-18
sale_gr1         2.661708e-17
capex_abn       -5.540096e-18
log_me           4.124479e-16
netdebt_me       4.224900e-17
o_score          3.074105e-18
z_score          1.100236e-16
eqnetis_at       5.027506e-17
log_age          4.193039e-16
inv_gr1         -5.549050e-18
dsale_dinv       5.685933e-18
kz_index         4.159717e-17
niq_su          -2.426689e-18
ni_inc8q        -4.793127e-18
dtype: float64


In [126]:
std_pre_test = 0
for x in test_update_window.keys():
    std_pre_test += test_update_window[x][0][model_features].std(ddof=0)

print("Train-Test Set Std:\n", std_pre_test/len(test_update_window), sep='')

Train-Test Set Std:
ret_2_0          1.0
ret_12_0         1.0
ret_24_12        1.0
resff3_12_1      1.0
aliq_at          1.0
ami_126d         1.0
turnover_126d    1.0
bidaskhl_21d     1.0
ivol_capm_21d    1.0
ivol_ff3_21d     1.0
rvol_21d         1.0
beta_252d        1.0
gp_at            1.0
ocf_at           1.0
qmj              1.0
f_score          1.0
be_me            1.0
at_gr1           1.0
sale_gr1         1.0
capex_abn        1.0
log_me           1.0
netdebt_me       1.0
o_score          1.0
z_score          1.0
eqnetis_at       1.0
log_age          1.0
inv_gr1          1.0
dsale_dinv       1.0
kz_index         1.0
niq_su           1.0
ni_inc8q         1.0
dtype: float64


In [127]:
for x in test_update_window.keys():
    print('Shape of train-test set', x,':', test_update_window[x][0].shape,'/', test_update_window[x][1].shape)

Shape of train-test set 2019-01-01 00:00:00 : (649829, 34) / (22387, 34)
Shape of train-test set 2019-07-01 00:00:00 : (672216, 34) / (22564, 34)
Shape of train-test set 2020-01-01 00:00:00 : (694780, 34) / (22970, 34)
Shape of train-test set 2020-07-01 00:00:00 : (717750, 34) / (23690, 34)
Shape of train-test set 2021-01-01 00:00:00 : (741440, 34) / (25377, 34)
Shape of train-test set 2021-07-01 00:00:00 : (766817, 34) / (26983, 34)
Shape of train-test set 2022-01-01 00:00:00 : (793800, 34) / (27618, 34)
Shape of train-test set 2022-07-01 00:00:00 : (821418, 34) / (27665, 34)
Shape of train-test set 2023-01-01 00:00:00 : (849083, 34) / (26537, 34)
Shape of train-test set 2023-07-01 00:00:00 : (875620, 34) / (25402, 34)
Shape of train-test set 2024-01-01 00:00:00 : (901022, 34) / (24482, 34)
Shape of train-test set 2024-07-01 00:00:00 : (925504, 34) / (19647, 34)


In [128]:
print("Inner Train Set Mean:\n",inner_train_df[model_features].mean(), sep='')
print("Inner Validation Set Mean\n",inner_valid_df[model_features].mean(), sep='')

Inner Train Set Mean:
ret_2_0         -1.956183e-17
ret_12_0        -1.600513e-17
ret_24_12        7.950698e-17
resff3_12_1      2.637883e-17
aliq_at         -2.155506e-16
ami_126d        -4.505149e-17
turnover_126d   -1.351545e-16
bidaskhl_21d     2.797935e-16
ivol_capm_21d   -4.078345e-16
ivol_ff3_21d    -2.575641e-16
rvol_21d         5.927827e-18
beta_252d        6.164940e-16
gp_at           -2.193296e-17
ocf_at          -2.548966e-17
qmj              5.594387e-18
f_score         -1.653864e-16
be_me            2.225899e-16
at_gr1          -5.631436e-17
sale_gr1         6.657691e-17
capex_abn        7.113393e-18
log_me          -1.659792e-17
netdebt_me       1.630153e-17
o_score          1.985822e-17
z_score          4.001283e-17
eqnetis_at      -1.763529e-17
log_age          5.145354e-16
inv_gr1          2.467458e-17
dsale_dinv       1.415269e-17
kz_index        -4.149479e-17
niq_su          -1.659792e-17
ni_inc8q         3.971644e-17
dtype: float64
Inner Validation Set Mean
ret_2_0

In [129]:
print("Inner Trainning Set Std:\n",inner_train_df[model_features].std(ddof=0), sep='')
print("Inner Validation Set Std\n",inner_valid_df[model_features].std(ddof=0), sep='')

Inner Trainning Set Std:
ret_2_0          1.0
ret_12_0         1.0
ret_24_12        1.0
resff3_12_1      1.0
aliq_at          1.0
ami_126d         1.0
turnover_126d    1.0
bidaskhl_21d     1.0
ivol_capm_21d    1.0
ivol_ff3_21d     1.0
rvol_21d         1.0
beta_252d        1.0
gp_at            1.0
ocf_at           1.0
qmj              1.0
f_score          1.0
be_me            1.0
at_gr1           1.0
sale_gr1         1.0
capex_abn        1.0
log_me           1.0
netdebt_me       1.0
o_score          1.0
z_score          1.0
eqnetis_at       1.0
log_age          1.0
inv_gr1          1.0
dsale_dinv       1.0
kz_index         1.0
niq_su           1.0
ni_inc8q         1.0
dtype: float64
Inner Validation Set Std
ret_2_0          0.846743
ret_12_0         0.884632
ret_24_12        0.812295
resff3_12_1      1.008161
aliq_at          1.143077
ami_126d         0.973512
turnover_126d    0.914760
bidaskhl_21d     0.765617
ivol_capm_21d    0.878423
ivol_ff3_21d     0.869696
rvol_21d         0.79721

In [130]:
print("Inner Trainning Set:",inner_train_df.shape)
print("Inner Validation Set:",inner_valid_df.shape)

Inner Trainning Set: (383570, 34)
Inner Validation Set: (133564, 34)


# Model Estimation

This section estimates all benchmark and machine-learning models required for the project. Following the project guideline, we compare at least:
- a historical-average benchmark,
- a pooled OLS baseline,
- and a machine-learning model (pooled Elastic Net).

Our main machine-learning direction is a cluster-conditional prediction model:
1. firms are grouped into economically meaningful clusters using k-means,
2. predictor effects are then allowed to differ across clusters through cluster dummies and predictor-by-cluster interactions,
3. Elastic Net is used both for first-stage predictor screening and for the final clustered prediction model.

All hyperparameters are chosen in the inner loop only, then held fixed afterward. During the outer-validation and test forecasting stages, we update preprocessing objects, cluster centroids, and model coefficients every 6 months using an expanding window.

## Historical-average benchmark

### Objective
Estimate the required benchmark model from the project guideline.

### Model idea
For each stock, predict next-month excess return as that stock’s own historical average excess return, using only information available up to the forecast date.

### What we do
- At each 6-month refit date, use all available past observations for each stock.
- Compute each stock’s historical mean of `ret_exc_lead1m`.
- Use that mean as the stock’s forecast for the next 6 months.

### Why we do it
- This is the required benchmark in the project brief.
- It provides a simple stock-level baseline before introducing cross-sectional characteristics or ML structure.

### Output
- Monthly stock-level predictions in the validation period
- Monthly stock-level predictions in the test period

## OLS Baseline

### Objective
Estimate the pooled linear benchmark model without regularization or cluster structure.

### Model idea
Use the retained predictor set in a pooled cross-sectional OLS regression to predict next-month excess returns.

### Inputs
- Preprocessed retained predictor set from the first-stage screening step (?)
- No cluster dummies
- No interactions

### What we do
- At each 6-month refit date, fit pooled OLS on the current expanding estimation window.
- Use the fitted model to predict stock-level returns for the next 6 months.

### Why we do it
- This is the required linear baseline in the project brief.
- It shows how much predictive power is available in a simple pooled specification before adding regularization or cluster-conditioned structure.

### Output
- Monthly stock-level predictions in the validation period
- Monthly stock-level predictions in the test period

## Elastic Net

### Objective
Use Elastic Net in two roles:
1. predictor screening,
2. pooled regularized benchmark estimation.

### Part A. Predictor screening (?)
We begin with a theory-motivated candidate predictor set covering:
- momentum,
- liquidity/frictions,
- risk/volatility,
- profitability/quality,
- value/investment,
- distress/issuance.

Confirmed candidate predictors include:
- Momentum: `ret_2_0`, `ret_12_0`, `ret_24_12`, `resff3_12_1`
- Liquidity/Frictions: `aliq_at`, `ami_126d`, `turnover_126d`, `bidaskhl_21d`
- Risk/Volatility: `ivol_capm_21d`, `ivol_ff3_21d`, `rvol_21d`, `beta_252d`
- Profitability/Quality: `gp_at`, `ocf_at`, `qmj`, `f_score`
- Value/Investment: `be_me`, `at_gr1`, `sale_gr1`, `capex_abn`, `me`
- Distress/Issuance: `netdebt_me`, `o_score`, `z_score`, `eqnetis_at`

Elastic Net is first fit on the inner training sample and evaluated on the inner validation sample. Predictors with useful non-zero contribution are retained for the downstream models.

### Part B. Pooled Elastic Net benchmark
After screening, we also estimate a pooled Elastic Net benchmark using the retained predictor set only.

### Why we do it
- Elastic Net handles correlated stock characteristics more stably than pure Lasso.
- Screening reduces dimensionality before building the much larger clustered interaction design.
- The pooled Elastic Net benchmark helps distinguish the effect of regularization from the effect of clustering.

### Hyperparameters
- Tune `alpha` and `lambda` in the inner loop only.
- Keep the chosen values fixed afterward.

### Output
- Retained predictor set
- Pooled Elastic Net predictions in validation and test

## Clustering: K-means

### Objective
Cluster firms into economically meaningful groups before estimating the cluster-conditional prediction model.

### Clustering variables
We use a smaller firm-type variable set, distinct from the prediction set, though overlapping is allowed.

Confirmed clustering variables:
- Lifecycle: `age`
- Profitability–Investment structure: `gp_at`, `ocf_at`, `sale_gr1`, `inv_gr1`, `dsale_dinv`
- Financial fragility: `netdebt_me`, `kz_index`, `z_score`
- Information environment: `niq_su`, `ni_inc8q`

### Algorithm
- k-means clustering
- Candidate number of clusters: `K = 2, 3, 4, 5, ...` (tune K)

### What we do
- In the inner loop, fit k-means on the inner training sample for each candidate `K`.
- Assign inner-validation observations to the nearest centroids.
- Use inner-validation performance to help choose the final `K`.

### After tuning
- Keep the chosen `K*` fixed.
- Re-estimate centroids every 6 months in the outer-validation and test forecasting loops using the current expanding estimation window.

### Output
- Cluster assignments for each stock-month
- Centroids for interpretation

## Clustered Elastic Net

### Objective
Estimate the main model of the project: a cluster-conditional return-prediction model.

### Model idea
The final design matrix includes:
- retained base predictors,
- cluster dummies,
- predictor-by-cluster interactions.

This allows the predictive effect of stock characteristics to vary across economically different firm clusters.

### What we do
1. Use k-means to assign each stock-month to a cluster.
2. Create `K-1` cluster dummies.
3. Interact each retained predictor with the cluster dummies.
4. Fit Elastic Net on this expanded design.

### Why Elastic Net
- The interaction design is high-dimensional.
- Base predictors and their interaction terms are highly correlated.
- Elastic Net combines selection from the L1 penalty with stabilization from the L2 penalty.

### Hyperparameters
- Tune `K`, `alpha`, and `lambda` in the inner loop only.
- Keep them fixed afterward.

### Output
- Clustered stock-level predictions in validation and test
- Estimated active interaction terms for interpretation

## Inner Tuning and Model Selection

### Objective
Use the inner split to finalize the hyperparameters and model setup before entering the official outer-validation period.

### Inner split
- Inner train: 2005-01 to 2012-12
- Inner validation: 2013-01 to 2015-12

### What is tuned
- Number of clusters `K`
- Elastic Net `alpha`
- Elastic Net `lambda`

### What we do
1. Run pooled Elastic Net screening on the candidate predictor set.
2. For each candidate `K`, fit k-means on the inner training sample.
3. For each candidate combination of `K`, `alpha`, and `lambda`, estimate the clustered Elastic Net model.
4. Evaluate performance on the inner validation sample.
5. Select:
   - the retained predictor set,
   - the final `K*`,
   - the final `alpha*`,
   - the final `lambda*`.

### Why we do it
Using an inner loop prevents the official 2016–2018 validation period from being heavily used for hyperparameter search, which keeps outer-validation model comparison cleaner.

# Model Evaluation and Comparison

This section compares all finalized models under a realistic forecasting protocol. We use the same 6-month expanding-window updating rule in both the outer-validation and test set so that the comparison is fair and consistent across models.

## Validation-Period Comparison

### Objective
Compare the finalized models on the official validation period before moving to the untouched test period.

### Validation period
- 2016-01 to 2018-12

### Models compared
- Historical-average benchmark
- Pooled OLS
- Pooled Elastic Net
- Clustered Elastic Net

### What we do
- Start from the hyperparameters chosen in the inner loop.
- Apply the 6-month expanding-window refit protocol.
- Generate monthly validation-period predictions for all models.
- Compare predictive performance across models.

### Main questions
- Does regularization improve performance relative to pooled OLS?
- Does cluster-conditioning improve performance relative to pooled Elastic Net?
- Is the chosen design stable enough to carry into the test period?

## 6-month Updating

### Objective
Implement the agreed real-time forecasting protocol.

### Update rule
- Expanding window
- Refit every 6 months

### What updates every 6 months
- preprocessing objects,
- k-means centroids,
- model coefficients.

### What remains fixed
- clustering algorithm = k-means,
- chosen `K*`,
- chosen `alpha*`,
- chosen `lambda*`,
- retained predictor set,
- model structure.

### Why we do it
This lets the model adapt to newly available data while preserving a fixed model design and avoiding repeated hyperparameter re-optimization.

## Test Evaluation

### Objective
Evaluate final out-of-sample predictive performance on the official test period.

### Test period
- 2019-01 to 2024-12

### What we do
- Use the same 6-month expanding-window updating procedure as in the validation stage.
- Generate monthly stock-level predictions for each model.
- Compute out-of-sample R^2 on the test period.

### Required predictive output
Report test out-of-sample R^2 for:
- Historical average
- Pooled OLS
- Pooled Elastic Net
- Clustered Elastic Net

### Interpretation
If R^2 is small or negative, discuss why. Monthly stock-return prediction is noisy, and realistic out-of-sample R^2 values are often small.

## Portfolio Construction and Performance (?)

### Objective
Translate monthly return predictions into investment signals and evaluate economic performance.

### Portfolio rule
For each test-period month:
- rank stocks by predicted return,
- long the top decile,
- short the bottom decile,
- equal-weight within each leg,
- impose dollar neutrality.

### What we do
- Construct monthly long-short portfolio returns for each predictive model.
- Compare economic performance across models.

### Required portfolio output
Report:
- annualized mean excess return,
- annualized volatility,
- Sharpe ratio.

### Also discuss
- turnover,
- transaction costs,
- liquidity constraints,
- whether the strategy remains plausible after implementation frictions.

## Cluster Interpretation and Diagnostics

### Objective
Interpret the selected clustering structure and explain the economic meaning of the final model.

### What we do
- Summarize the final cluster centroids using the clustering variables.
- Give each cluster an economic interpretation.
- Examine which predictors survive the Elastic Net screening step.
- Examine which predictor-by-cluster interaction terms remain active in the final clustered Elastic Net model.

### Why this matters
This is what turns the project from a generic ML exercise into a finance project. We want to understand not only whether the clustered model performs better, but also whether the resulting firm groups and conditional predictor effects make economic sense.

### Key interpretation questions
- What type of firms does each cluster represent?
- Which predictors are important overall?
- Which predictors matter only in certain clusters?
- Does the clustered model tell a coherent economic story?